In [88]:
import pandas as pd
import norm
import numpy as np
import bintrans
import dfgen
import torch
import model_forward
import matplotlib.pyplot as plt
import os

In [89]:
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [90]:
param_path = '../../bat_mod/result/param.xlsx'
norm_path = '../trained_model/model5/train/norm_dict.npy'
model_path = '../trained_model/model5/train/my_model.pth'
xy_path = '../trained_model/model5/train/xy_dict.npy'

In [91]:
#生成输入文件
file = open('1(1).log', 'r')

In [92]:
data = file.readlines()

In [93]:
data

['Sat Jan 20 19:25:55 CST 2024\n',
 '\n',
 'Device Version Info = 1746_0_01\n',
 'BQZ Device Name = bq27z746\n',
 'BQZ Firmware Version = 0_01\n',
 '\n',
 '\n',
 'Sample,DateTime,ElapsedTime,Voltage,Current,LogRowTime(ms),LogStatus\n',
 '1,2024-01-20 19:25:56,1.013,3906,-1060,35,SUCCESS\n',
 '2,2024-01-20 19:25:57,2.027,3907,-2859,38,SUCCESS\n',
 '3,2024-01-20 19:25:58,3.034,3906,-2859,30,SUCCESS\n',
 '4,2024-01-20 19:25:59,4.046,3906,-2915,36,SUCCESS\n',
 '5,2024-01-20 19:26:00,5.060,3896,-3478,36,SUCCESS\n',
 '6,2024-01-20 19:26:01,6.071,3896,-3459,27,SUCCESS\n',
 '7,2024-01-20 19:26:02,7.084,3895,-3459,27,SUCCESS\n',
 '8,2024-01-20 19:26:03,8.093,3895,-3452,35,SUCCESS\n',
 '9,2024-01-20 19:26:04,9.104,3894,-3444,21,SUCCESS\n',
 '10,2024-01-20 19:26:05,10.117,3894,-3437,26,SUCCESS\n',
 '11,2024-01-20 19:26:06,11.126,3894,-3433,33,SUCCESS\n',
 '12,2024-01-20 19:26:07,12.138,3893,-3422,37,SUCCESS\n',
 '13,2024-01-20 19:26:08,13.148,3893,-3422,24,SUCCESS\n',
 '14,2024-01-20 19:26:09,14.

In [94]:
data[8]

'1,2024-01-20 19:25:56,1.013,3906,-1060,35,SUCCESS\n'

In [95]:
do = 0
u = ''
i = ''
for j in range(len(data[8])):
    if data[8][j] == ',':
        do = do + 1
    else:
        if do == 3:
            u = u + data[8][j]
        elif do==4:
            i = i + data[8][j]

In [96]:
u

'3906'

In [97]:
i

'-1060'

In [98]:
u_lis = []
i_lis = []
for i in range(9, 69):
    do = 0
    u = ''
    c = ''
    for j in range(len(data[i])):
        if data[i][j] == ',':
            do = do + 1
        else:
            if do == 3:
                u = u + data[i][j]
            elif do==4:
                c = c + data[i][j]
    u_lis.append(u)
    i_lis.append(c)

In [99]:
u_int = []
for u in u_lis:
    u_int.append(int(u))

In [100]:
i_int = []
for i in i_lis:
    i_int.append(int(i))

In [101]:
ui = np.array([u_int, i_int]).T
iu = np.array([i_int, u_int]).T

In [102]:
ui.shape

(60, 2)

In [103]:
def mat2bintxt(param_scal):
    #将矩阵每一行的所有列合并为字符串
    #每一列为16位2进制数
    #返回列表
    param_list = []
    for i in range(param_scal.shape[0]):
        new_bin = ''
        for j in range(param_scal.shape[1]):
            if (j==(param_scal.shape[1]-1)):
                param_bin = bintrans.dec2bnr(int(param_scal[i, j])) + '\n'
            else:
                param_bin = bintrans.dec2bnr(int(param_scal[i, j]))
            new_bin = new_bin + param_bin
        param_list.append(new_bin)
    return param_list

In [104]:
inpu = mat2bintxt(ui)

In [105]:
inpu[30]

'00001111001100001111001011101111\n'

In [106]:
with open('input.txt', 'w') as f:
    f.writelines(inpu)

In [107]:
iu.reshape(2, 30, 2)

array([[[-2859,  3907],
        [-2859,  3906],
        [-2915,  3906],
        [-3478,  3896],
        [-3459,  3896],
        [-3459,  3895],
        [-3452,  3895],
        [-3444,  3894],
        [-3437,  3894],
        [-3433,  3894],
        [-3422,  3893],
        [-3422,  3893],
        [-3415,  3893],
        [-3408,  3892],
        [-3408,  3892],
        [-3400,  3892],
        [-3393,  3891],
        [-3393,  3891],
        [-3389,  3891],
        [-3386,  3891],
        [-3378,  3890],
        [-3375,  3890],
        [-3371,  3890],
        [-3367,  3890],
        [-3364,  3889],
        [-3364,  3889],
        [-3356,  3889],
        [-3352,  3889],
        [-3352,  3888],
        [-3349,  3888]],

       [[-3345,  3888],
        [-3341,  3888],
        [-3334,  3888],
        [-3338,  3888],
        [-3330,  3887],
        [-3330,  3887],
        [-3327,  3887],
        [-3323,  3887],
        [-3319,  3886],
        [-3319,  3886],
        [-3316,  3886],
        [-3312

In [108]:
param = pd.read_excel(param_path, header=None).values
norm_dict = np.load(norm_path, allow_pickle=True).item()
model = torch.load(model_path)

In [109]:
cap = 12000
scale = 2**12

In [110]:
input_dict = {'iu': iu, 'soc_ini': 0.92, 'param':param, 'cap': cap}

In [111]:
x_in = dfgen.dfgen(input_dict)
x_nm = norm.norm(x_in, norm_dict)

In [25]:
x_int = np.floor(x_nm * scale).reshape(2, 30, 8)

In [113]:
x_nm = x_nm.reshape(2, 30, 8)

[[[ 0.81459459 -0.2859      0.91993382  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.81405405 -0.2859      0.91986764  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.81405405 -0.2915      0.91980016  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80864865 -0.3478      0.91971965  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80864865 -0.3459      0.91963958  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80810811 -0.3459      0.91955951  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80810811 -0.3452      0.91947961  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80756757 -0.3444      0.91939988  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80756757 -0.3437      0.91932032  0.86394173  0.1437212
    0.10046626  0.03759826  0.04134325]
  [ 0.80756757 -0.3433      0.91924086  0.86394173  0.1437212
    0.10046626  0.03

In [114]:
parameters = {}
for name, param in model.named_parameters():
    parameters[name] = param.detach().numpy()

In [119]:
vlstm = model_forward.Vlstm(x_nm[0, :, :], parameters, scale)
plstm = model_forward.Plstm(x_nm[0, :, :], parameters)

In [120]:
result_v = vlstm.forward(30)
result_p = plstm.forward(30)

时间步：0
时间步：1
时间步：2
时间步：3
时间步：4
时间步：5
时间步：6
时间步：7
时间步：8
时间步：9
时间步：10
时间步：11
时间步：12
时间步：13
时间步：14
时间步：15
时间步：16
时间步：17
时间步：18
时间步：19
时间步：20
时间步：21
时间步：22
时间步：23
时间步：24
时间步：25
时间步：26
时间步：27
时间步：28
时间步：29
时间步：0
时间步：1
时间步：2
时间步：3
时间步：4
时间步：5
时间步：6
时间步：7
时间步：8
时间步：9
时间步：10
时间步：11
时间步：12
时间步：13
时间步：14
时间步：15
时间步：16
时间步：17
时间步：18
时间步：19
时间步：20
时间步：21
时间步：22
时间步：23
时间步：24
时间步：25
时间步：26
时间步：27
时间步：28
时间步：29


In [136]:
result_v['result']

array([4244.], dtype=float32)

In [139]:
4175 / scale

1.019287109375

In [122]:
result_p['result']

array([0.99564755], dtype=float32)

In [32]:
xy = np.load(xy_path, allow_pickle=True).item()

In [123]:
xy['train_x'][0, :, :]

array([[0.97567568, 0.0774    , 0.93164566, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97675676, 0.0772    , 0.93164594, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97297297, 0.0772    , 0.93164594, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97459459, 0.0769    , 0.93164636, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97567568, 0.077     , 0.93164622, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97243243, 0.0766    , 0.93164678, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97513514, 0.0766    , 0.93164678, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97891892, 0.0763    , 0.93164719, 0.86394173, 0.1437212 ,
        0.10046626, 0.03759826, 0.04134325],
       [0.97405405, 0.0762    , 0.93164733, 0.86394173, 0.1437212 ,
        0.10046626, 0.037598

In [124]:
train_x = xy['train_x'][0, :, :]

In [133]:
x_ts = torch.from_numpy(train_x.reshape(-1, 30, 8)).type(torch.float32)
x_cj_ts = torch.from_numpy(x_nm.reshape(-1, 30, 8)).type(torch.float32)

In [134]:
model(x_ts)

tensor(0.9258, grad_fn=<SqueezeBackward0>)

In [135]:
model(x_cj_ts)

tensor([0.9956, 0.9904], grad_fn=<SqueezeBackward0>)

In [137]:
x_ts

tensor([[[0.9757, 0.0774, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9768, 0.0772, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9730, 0.0772, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9746, 0.0769, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9757, 0.0770, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9724, 0.0766, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9751, 0.0766, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9789, 0.0763, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9741, 0.0762, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9751, 0.0761, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9724, 0.0760, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9757, 0.0758, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9730, 0.0755, 0.9316, 0.8639, 0.1437, 0.1005, 0.0376, 0.0413],
         [0.9714, 0.0754,

In [140]:
x_int

array([[[ 3336., -1172.,  3768.,  3538.,   588.,   411.,   154.,   169.],
        [ 3334., -1172.,  3767.,  3538.,   588.,   411.,   154.,   169.],
        [ 3334., -1194.,  3767.,  3538.,   588.,   411.,   154.,   169.],
        [ 3312., -1425.,  3767.,  3538.,   588.,   411.,   154.,   169.],
        [ 3312., -1417.,  3766.,  3538.,   588.,   411.,   154.,   169.],
        [ 3310., -1417.,  3766.,  3538.,   588.,   411.,   154.,   169.],
        [ 3310., -1414.,  3766.,  3538.,   588.,   411.,   154.,   169.],
        [ 3307., -1411.,  3765.,  3538.,   588.,   411.,   154.,   169.],
        [ 3307., -1408.,  3765.,  3538.,   588.,   411.,   154.,   169.],
        [ 3307., -1407.,  3765.,  3538.,   588.,   411.,   154.,   169.],
        [ 3305., -1402.,  3764.,  3538.,   588.,   411.,   154.,   169.],
        [ 3305., -1402.,  3764.,  3538.,   588.,   411.,   154.,   169.],
        [ 3305., -1399.,  3764.,  3538.,   588.,   411.,   154.,   169.],
        [ 3303., -1396.,  3763.,  3538